# 01b · Target-focused High-Gamma extraction

只载入 20 mm 敏感性范围内候选触点及其 Laplacian 邻居，避免对无关全脑通道重复做 8 个子带的 Hilbert。

In [1]:
# [Setup]
from pathlib import Path
import sys, ast, numpy as np, pandas as pd, mne
ROOT=Path('/home/lirui/liulab_project/ieeg/Project_colorieeg_2026'); PIPE=ROOT/'color_cognition_pipeline'/'analyse_0720'
sys.path.insert(0,str(PIPE)); import config
from utils.preprocessing import add_laplacian_neighbors, contact_laplacian
from utils.epochs import save_epochs, baseline_zscore

In [2]:
# [Targets] Primary candidates plus 20-mm sensitivity set
loc=pd.read_excel(ROOT/'processed_data'/'test001'/'test001_ieegloc.xlsx')
channel_col=next(c for c in loc.columns if c.lower() in ('channel','channelname','name'))
def parse_mni(x):
    try:return np.asarray(ast.literal_eval(str(x)),float)
    except Exception:return np.full(3,np.nan)
loc['coord']=loc['MNI'].map(parse_mni); targets=[np.asarray(x) for x in config.FMRI_TARGETS.values()]
loc['target_distance_mm']=loc['coord'].map(lambda p:min(np.linalg.norm(p-t) for t in targets) if np.isfinite(p).all() else np.nan)
target_labels=loc.loc[loc.target_distance_mm<=max(config.SENSITIVITY_RADII_MM),channel_col].drop_duplicates().tolist()
print('target/sensitivity contacts:',target_labels)

target/sensitivity contacts: ['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'B1', 'B2', 'B3', 'B4', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'G3', 'G4', 'G5', 'G6', 'G7', 'G8', 'G9']


In [3]:
# [Extract] Process one run at a time and cache the finished HG epochs
def process_hg(subject,task,set_path):
    raw=mne.io.read_raw_eeglab(set_path,preload=True,verbose=False)
    picks=add_laplacian_neighbors(raw.ch_names,[c for c in target_labels if c in raw.ch_names]); raw.pick(picks)
    raw.notch_filter(config.LINE_NOISE_HZ,verbose=False); ref=contact_laplacian(raw,target_labels)
    events,event_id=mne.events_from_annotations(ref,verbose=False); selected={n:c for n,c in event_id.items() if n.startswith('Trigger-In:')}
    accum=np.zeros(ref.get_data().shape,dtype=np.float32)
    for lo,hi in config.HG_BANDS_HZ:
        band=ref.copy().filter(lo,hi,verbose=False).apply_hilbert(envelope=True,verbose=False)
        power=np.square(band.get_data()); accum += (np.log10(np.maximum(power,np.finfo(float).eps))/len(config.HG_BANDS_HZ)).astype(np.float32)
        del band,power
    hg=mne.io.RawArray(accum,mne.create_info(ref.ch_names,ref.info['sfreq'],'seeg'),verbose=False); hg.set_annotations(ref.annotations.copy())
    epochs=mne.Epochs(hg,events,event_id=selected,tmin=config.EPOCH_TMIN_S,tmax=config.EPOCH_TMAX_S,baseline=None,preload=True,reject_by_annotation=False,verbose=False)
    data=baseline_zscore(epochs.get_data(),epochs.times*1000,tuple(v*1000 for v in config.HG_BASELINE_S))
    inverse={c:n for n,c in selected.items()}; triggers=[inverse[int(c)] for c in epochs.events[:,2]]
    out=config.INTERMEDIATE_ROOT/subject/'preprocessing'/f'task{task}_hg.npz'
    save_epochs(out,data,epochs.times*1000,triggers,epochs.ch_names,{'task':task,'reference':'contact_laplacian','bands_hz':config.HG_BANDS_HZ,'baseline_ms':[-250,-50],'source':str(set_path)})
    del raw,ref,accum,hg,epochs,data

In [4]:
# [Run] Sequential runs bound peak memory; change SUBJECTS in config for future subjects
for task,run_name in config.RUNS.items(): process_hg('test001',task,config.subject_raw_dir('test001')/f'{run_name}.set')
print('High-Gamma extraction complete')

High-Gamma extraction complete
